# Lab 01 — Um star schema na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: consultar um **esquema estrela** (fato + dimensões) e sentir por que ele é simples de analisar.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
con.execute("CREATE TABLE dim_cliente(cliente_id INT, nome VARCHAR, cidade VARCHAR)")
con.executemany("INSERT INTO dim_cliente VALUES (?,?,?)", [(1,'ana','São Paulo'),(2,'bruno','Rio de Janeiro'),(3,'caio','Belo Horizonte')])
con.execute("CREATE TABLE dim_produto(produto_id INT, categoria VARCHAR)")
con.executemany("INSERT INTO dim_produto VALUES (?,?)", [(10,'eletronicos'),(20,'livros'),(30,'casa')])
con.execute("CREATE TABLE fato_vendas(venda_id INT, cliente_id INT, produto_id INT, quantidade INT, valor DOUBLE)")
con.executemany("INSERT INTO fato_vendas VALUES (?,?,?,?,?)", [
    (1,1,10,1,1200.0),(2,2,20,2,50.0),(3,1,20,1,30.0),(4,3,30,1,80.0),
    (5,2,10,1,800.0),(6,1,10,1,1500.0),(7,3,20,4,35.0),(8,2,30,1,110.0)])
con.execute('SELECT COUNT(*) FROM fato_vendas').fetchone()

## 1. Juntar o fato com as dimensões
A análise vira: fato + dimensões que interessam → agrega.

In [ ]:
con.execute('''
    SELECT c.cidade, p.categoria, SUM(f.valor) AS receita
    FROM fato_vendas f
    JOIN dim_cliente c ON f.cliente_id = c.cliente_id
    JOIN dim_produto p ON f.produto_id = p.produto_id
    GROUP BY c.cidade, p.categoria
    ORDER BY receita DESC
''').df()

## 2. Sua vez (mini-desafio)
Traga a **receita por categoria** (colunas `categoria`, `receita`), da maior para a menor, juntando o fato com `dim_produto`. Verifique.

In [ ]:
resposta = con.execute('''
    SELECT p.categoria, SUM(f.valor) AS receita
    FROM fato_vendas f
    JOIN dim_produto p ON f.produto_id = p.produto_id
    GROUP BY p.categoria
    ORDER BY receita DESC
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    try:
        assert rows == [('eletronicos',3500.0),('casa',190.0),('livros',115.0)], 'Confira o JOIN e o GROUP BY.'
        print('\u2705 Correto! Você consultou o star schema (fato + dimensão).')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)